## Importing Libraries

In [42]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json

## Setting up files

In [43]:
GENERATION_MODEL = "qwen3:1.7b"
FILES_REP = glob.glob("../Test_Files/Eligibility_reports/eligibility-rep_*.json")
FILE_PROF = "../Test_Files/Patient_profiles/patient-profiles.json"
FILE_RULES = "../Test_Files/clinical-trial-extracted_e1.txt"

PROMPT_FILE = "./prompts/justification-generation_prompt.txt"
OUTPUT_DIR = "./llm-outputs/justification-generation/"
OUTPUT_FILE = "experiment"

print(f"Found the following eligibility reports - {FILES_REP}")

Found the following eligibility reports - ['../Test_Files/Eligibility_reports\\eligibility-rep_e1.json']


## Setting up environment

In [44]:
## Setting evironment

with open(PROMPT_FILE,"r", encoding="utf-8") as p:
    prompt_arr = [t.strip() for t in p.readlines() if t.strip()]
    base_prompt = " ".join(prompt_arr)

os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isfile(os.path.join(OUTPUT_DIR, path)):
        count += 1

## Justification generation
In this phase the justification generation for a eligibility decision will be done by a LLM, it must have the patient profile and the logic rule converted trial criteria for a clear justification

In [45]:
for file in FILES_REP:
    with open(file, 'r') as f, open(FILE_PROF,'r') as patient_profiles, open(FILE_RULES,'r', encoding='utf-8') as trial_rules:
        
        data_rep = json.load(f)
        data_prof = json.load(patient_profiles)
        
        trial_rules_arr = [t.strip() for t in trial_rules.readlines() if t.strip()]
        trial_rules_text = " ".join(trial_rules_arr)
        
        pbar = tqdm(total=len(data_rep), desc="Processing eligibility entries")
        
        for entry in data_rep:
            patient_id = entry['Patient'].split('-')[0]
            for patient in data_prof:
                if patient_id == str(patient['patient_id']):
                    prompt_w_patient = base_prompt.replace("{{PATIENT_PROFILE}}",json.dumps(patient))
                    prompt_w_rules = prompt_w_patient.replace("{{LOGIC_CRITERIA}}",trial_rules_text)
                    prompt_final = prompt_w_rules.replace("{{ELIGIBLE_LIST}}",json.dumps(entry))
                    
                    print(prompt_final)
                    
                    stream = chat(
                        model=GENERATION_MODEL,
                        messages=[{"role": "user", "content": prompt_final}],
                        stream=True,
                        )
                    
                    llm_output = ""
                    for chunk in stream:
                        llm_output += chunk["message"]["content"]

                    with open(f"{OUTPUT_DIR}{OUTPUT_FILE}-{count}.txt","a",encoding="utf-8") as o:
                        o.write(f"Ouput for file {file}\n")
                        o.write(f"{llm_output}\n\n")
                        print(f"Saved LLM output on {OUTPUT_FILE}-{count}")
                        
                    
                    print("\n")
                    
                    pbar.update(1)
                
        pbar.close()
        
        

Processing eligibility entries:   0%|          | 0/3 [00:00<?, ?it/s]

You are an assistant responsible for generating a natural‑language justification for a clinical trial eligibility decision. You will receive: - A patient profile with structured clinical information. - A list of clinical trial eligibility criteria expressed as logic statements. - An eligibility decision object in the form: { "Patient": "<patient_id>-<Name>", "Decision": 1 or 0 } Your task is to produce a continuous, coherent natural‑language explanation describing why the patient was classified as eligible (Decision = 1) or not eligible (Decision = 0). Instructions: - Explain how the patient’s data aligns or conflicts with each criterion. - Mention which criteria were met, which were not met, and which could not be evaluated because their logic was null. - Base the explanation strictly on the provided patient data and logic expressions. - Do not infer or assume missing information. - Do not add clinical interpretation beyond what is explicitly stated. - The justification must be consis

Processing eligibility entries:  33%|███▎      | 1/3 [00:46<01:33, 46.59s/it]

Saved LLM output on experiment-2


You are an assistant responsible for generating a natural‑language justification for a clinical trial eligibility decision. You will receive: - A patient profile with structured clinical information. - A list of clinical trial eligibility criteria expressed as logic statements. - An eligibility decision object in the form: { "Patient": "<patient_id>-<Name>", "Decision": 1 or 0 } Your task is to produce a continuous, coherent natural‑language explanation describing why the patient was classified as eligible (Decision = 1) or not eligible (Decision = 0). Instructions: - Explain how the patient’s data aligns or conflicts with each criterion. - Mention which criteria were met, which were not met, and which could not be evaluated because their logic was null. - Base the explanation strictly on the provided patient data and logic expressions. - Do not infer or assume missing information. - Do not add clinical interpretation beyond what is explicitly stated.

Processing eligibility entries:  67%|██████▋   | 2/3 [01:47<00:54, 54.80s/it]

Saved LLM output on experiment-2


You are an assistant responsible for generating a natural‑language justification for a clinical trial eligibility decision. You will receive: - A patient profile with structured clinical information. - A list of clinical trial eligibility criteria expressed as logic statements. - An eligibility decision object in the form: { "Patient": "<patient_id>-<Name>", "Decision": 1 or 0 } Your task is to produce a continuous, coherent natural‑language explanation describing why the patient was classified as eligible (Decision = 1) or not eligible (Decision = 0). Instructions: - Explain how the patient’s data aligns or conflicts with each criterion. - Mention which criteria were met, which were not met, and which could not be evaluated because their logic was null. - Base the explanation strictly on the provided patient data and logic expressions. - Do not infer or assume missing information. - Do not add clinical interpretation beyond what is explicitly stated.

Processing eligibility entries: 100%|██████████| 3/3 [02:31<00:00, 50.43s/it]

Saved LLM output on experiment-2


